# Подборки туристических туров (Москва → Европа)

1. **Сбор данных** — API Travelpayouts (5 запросов)
2. **Парсинг** — Booking.com (Selenium)
3. **Очистка** — типы, пропуски
4. **Подборки** — дешёвые туры, лучшее соотношение цена/качество, сравнение городов

**Итоговая переменная:** `total_price` = цена перелёта + цена отеля (EUR)

In [ ]:
%pip install requests pandas selenium numpy -q

---
# Часть 1. Сбор данных о перелётах (API Travelpayouts)

In [ ]:
import os
import re
import csv
import time
import random
import requests
import numpy as np
import pandas as pd
from datetime import date, timedelta
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

API_TOKEN = "dbc5a45e021de9214f5865a36cbba36d"
HEADERS = {"Accept-Encoding": "gzip, deflate"}

ORIGIN = "MOW"
DEPARTURE_MONTH = "2026-07"
DEPARTURE_DATE = "2026-07-01"
RETURN_DATE = "2026-07-15"

CHECKIN = date(2026, 7, 1)
CHECKOUT = CHECKIN + timedelta(days=14)
ADULTS = 2
ROOMS = 1
PAGES_PER_CITY = 16

# ISO 3166-1 alpha-2 коды европейских стран
EUROPEAN_COUNTRY_CODES = { "AL", "AD", "AT", "BE", "BA", "BG", "HR", "CY", "CZ", "DK",
    "EE", "FI", "FR", "DE", "GR", "HU", "IS", "IE", "IT", "LV",
    "LI", "LT", "LU", "MT", "MD", "MC", "ME", "NL", "MK", "NO",
    "PL", "PT", "RO", "SM", "RS", "SK", "SI", "ES", "SE", "CH",
    "UA", "GB", "VA"}

RAW_DIR = "data/raw"
CLEAN_DIR = "data/clean"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)

request_counter = 0

In [ ]:
def api_get(url, params=None):
    response = requests.get(url, params=params, headers=HEADERS, timeout=30)
    response.raise_for_status()
    data = response.json()
    return data


def to_dataframe(data, key_name="key"):
    if data is None:
        return pd.DataFrame()
    if isinstance(data, list):
        return pd.DataFrame(data)
    if isinstance(data, dict):
        rows = []
        for k, v in data.items():
            if isinstance(v, dict):
                rows.append({key_name: k, **v})
            else:
                rows.append({key_name: k, "value": v})
        return pd.DataFrame(rows)
    return pd.DataFrame()

In [ ]:
# API 1: справочник городов
response = requests.get("https://api.travelpayouts.com/data/en/cities.json", timeout=30)
response.raise_for_status()
df_cities = pd.DataFrame(response.json())
print(f"Городов: {len(df_cities)}")

# API 2: справочник аэропортов
response = requests.get("https://api.travelpayouts.com/data/en/airports.json", timeout=30)
response.raise_for_status()
df_airports = pd.DataFrame(response.json())
print(f"Аэропортов: {len(df_airports)}")

df_airports[df_airports["city_code"] == ORIGIN][["code", "name", "city_code"]].head()

In [ ]:
# API 3: популярные направления из Москвы, фильтруем только Европу

params = {
    "origin": ORIGIN, "departure_at": DEPARTURE_MONTH,
    "currency": "eur", "limit": 100, "page": 1,
    "sorting": "route", "unique": "true",
    "market": "ru", "token": API_TOKEN,
}
df_all_dest = to_dataframe(
    api_get("https://api.travelpayouts.com/aviasales/v3/prices_for_dates", params).get("data", [])
)

if "destination" in df_all_dest.columns:
    df_all_dest = df_all_dest.rename(columns={"destination": "destination_code"})

# подтягиваем название города и страну из справочника
cities_info = df_cities[["code", "name", "country_code"]].rename(
    columns={"code": "destination_code", "name": "destination_name"}
)
df_all_dest = df_all_dest.merge(cities_info, on="destination_code", how="left")

# оставляем только европейские, топ-30 по цене
df_all_dest["country_code"] = df_all_dest["country_code"].astype(str).str.upper()
df_euro =df_all_dest[df_all_dest["country_code"].isin(EUROPEAN_COUNTRY_CODES)]
df_euro = df_euro.drop_duplicates(subset=["destination_code"])
df_euro = df_euro.sort_values("price").head(30).reset_index(drop=True)

print(f"Европейских направлений: {len(df_euro)}")
df_euro[["destination_code", "destination_name", "country_code", "price"]]

In [ ]:
# API 4: цены на билеты по конкретным датам для каждого направления (металофф пофиксит)

destinations = df_euro["destination_code"].tolist()
all_prices = []

for i, dest in enumerate(destinations, 1):
    params = {
        "origin": ORIGIN, "destination": dest,
        "departure_at": DEPARTURE_DATE, "return_at": RETURN_DATE,
        "currency": "eur", "limit": 30, "page": 1,
        "direct": "false", "one_way": "false",
        "sorting": "price", "market": "ru", "token": API_TOKEN,
    }
    df_p = to_dataframe(
        api_get("https://api.travelpayouts.com/aviasales/v3/prices_for_dates", params).get("data", [])
    )
    if not df_p.empty:
        df_p["requested_destination"] = dest
        all_prices.append(df_p)
        print(f"{dest}: {len(df_p)} записей")
    else:
        print(f"{dest}: нет данных")

df_prices = pd.concat(all_prices, ignore_index=True) if all_prices else pd.DataFrame()
print(f"Итого билетов: {len(df_prices)}")

In [ ]:
# API 5: минимальные цены, сгруппированные по дате вылета

all_grouped = []
for i, dest in enumerate(destinations, 1):
    params = {
        "origin": ORIGIN, "destination": dest,
        "departure_at": DEPARTURE_MONTH, "currency": "eur",
        "group_by": "departure_at", "direct": "false",
        "min_trip_duration": 14, "max_trip_duration": 14,
        "market": "ru", "token": API_TOKEN,
    }
    df_g = to_dataframe(
        api_get("https://api.travelpayouts.com/aviasales/v3/grouped_prices", params).get("data", {}),
        key_name="group_key"
    )
    if not df_g.empty:
        df_g["requested_destination"] = dest
        all_grouped.append(df_g)
        print(f"[{i}/{len(destinations)}] {dest}: {len(df_g)} дат")
    else:
        print(f"[{i}/{len(destinations)}] {dest}: нет данных")

df_grouped = pd.concat(all_grouped, ignore_index=True) if all_grouped else pd.DataFrame()
print(f"Итого grouped_prices: {len(df_grouped)}")

In [ ]:
datasets = {
    "api_cities.csv": df_cities,
    "api_airports.csv": df_airports,
    "api_popular_european_destinations.csv": df_euro,
    "api_prices_for_dates.csv": df_prices,
    "api_grouped_prices.csv": df_grouped,
}

for filename, df in datasets.items():
    path = os.path.join(RAW_DIR, filename)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{path} — {len(df)} rows")

---
# Часть 2. Парсинг отелей с Booking.com (Selenium)

In [ ]:
DEFAULT_CITIES = [
    "Rome", "Paris", "Barcelona", "London", "Amsterdam",
    "Prague", "Vienna", "Berlin", "Lisbon", "Athens",
    "Budapest", "Madrid", "Milan", "Dublin", "Warsaw",
    "Copenhagen", "Stockholm", "Helsinki", "Zurich", "Brussels",
    "Bucharest", "Sofia", "Zagreb", "Tallinn", "Riga"]

# берём города из API если получилось, иначе дефолтный список
if "destination_name" in df_euro.columns:
    CITIES = df_euro["destination_name"].dropna().tolist()
else:
    CITIES = DEFAULT_CITIES

# если из API пришло меньше 25 то сбда дополняем дефолтными
if len(CITIES) < 25:
    seen = set(c.lower() for c in CITIES)
    for c in DEFAULT_CITIES:
        if c.lower() not in seen:
            CITIES.append(c)
            seen.add(c.lower())
        if len(CITIES) >= 25:
            break

print(f"Городов: {len(CITIES)}")
print(CITIES)

In [ ]:
CSV_OUTPUT = os.path.join(RAW_DIR, "booking_hotels_all_cities.csv")
def get_text(parent, selector):
    els = parent.find_elements(By.CSS_SELECTOR, selector)
    return els[0].text.strip() if els else ""


def get_attr(parent, selector, attr):
    els = parent.find_elements(By.CSS_SELECTOR, selector)
    return els[0].get_attribute(attr) or "" if els else ""


def create_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--disable-notifications")
    options.add_argument("--lang=en-GB")
    options.add_argument("--window-size=1920,1080")

    options.add_experimental_option("prefs", {
        "profile.managed_default_content_settings.images": 2
    })
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(0)
    driver.set_page_load_timeout(30)
    return driver


def close_popups(driver):
    for selector in [
        'button[id*="onetrust-accept"]',
        'button[aria-label*="Accept"]',
        'button[aria-label="Close"]',
        'button[aria-label="Dismiss sign-in info."]',
    ]:
        btns = driver.find_elements(By.CSS_SELECTOR, selector)
        for btn in btns:
            if btn.is_displayed():
                btn.click()
                time.sleep(0.2)
                return


def build_url(city, offset=0):
    return (
        f"https://www.booking.com/searchresults.en-gb.html"
        f"?ss={city}"
        f"&checkin={CHECKIN.isoformat()}"
        f"&checkout={CHECKOUT.isoformat()}"
        f"&group_adults={ADULTS}"
        f"&no_rooms={ROOMS}"
        f"&group_children=0"
        f"&order=popularity"
        f"&offset={offset}"
    )


def parse_cards(driver, city):
    cards = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="property-card"]')
    results = []
    for card in cards:
        url = get_attr(card, 'a[data-testid="title-link"]', "href")
        if not url or "/searchresults" in url:
            continue
        results.append({
            "city": city,
            "hotel_name": get_text(card, '[data-testid="title"]'),
            "url": url,
            "distance_to_center": get_text(card, '[data-testid="distance"]'),
            "price": get_text(card, '[data-testid="price-and-discounted-price"]'),
            "rating_score": get_text(card, '[data-testid="review-score"] div'),
            "review_count": get_text(card, '[data-testid="review-score"] .abf093bdfe'),
        })
    return results


def scrape_city(driver, city):
    global request_counter
    all_hotels = []
    seen_urls = set()

    for page in range(PAGES_PER_CITY):
        driver.get(build_url(city, page * 25))
        request_counter += 1

        try:
            WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'div[data-testid="property-card"]'))
            )
        except:
            pass

        close_popups(driver)

        for _ in range(2):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(0.3)

        time.sleep(0.5)  # ждём пока JS прогрузит текст в карточках

        hotels = [h for h in parse_cards(driver, city) if h["url"] not in seen_urls]
        for h in hotels:
            seen_urls.add(h["url"])
        all_hotels.extend(hotels)

        print(f"{city} стр.{page+1}:+{len(hotels)}(всего {len(all_hotels)})")
        if not hotels:
            break

        time.sleep(random.uniform(0.3, 0.7))

    return all_hotels

In [ ]:
CSV_COLUMNS = ["city", "hotel_name", "url", "distance_to_center", "price",
               "rating_score", "review_count"]

with open(CSV_OUTPUT, "w", newline="", encoding="utf-8-sig") as f:
    csv.DictWriter(f, fieldnames=CSV_COLUMNS).writeheader()

driver = create_driver()
total_hotels = 0

for i, city in enumerate(CITIES, 1):
    print(f"{len(CITIES)}]{city} ")
    hotels = scrape_city(driver, city)

    if not hotels:
        continue

    with open(CSV_OUTPUT, "a", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        for hotel in hotels:
            writer.writerow({col: hotel.get(col, "") for col in CSV_COLUMNS})

    total_hotels += len(hotels)
    print(f"{len(hotels)}, всего: {total_hotels}")
    time.sleep(0.5)

driver.quit()
print(f"Готово: {total_hotels} отелей,{request_counter} запросов")

---
# Часть 3. Очистка данных

In [ ]:
df_flights = pd.read_csv(os.path.join(RAW_DIR, "api_prices_for_dates.csv"))
print(f"Сырых:{df_flights.shape}")

df_flights["price"] = pd.to_numeric(df_flights["price"], errors="coerce")
for col in ["departure_at", "return_at"]:
    if col in df_flights.columns:
        df_flights[col] = pd.to_datetime(df_flights[col], errors="coerce")

df_flights = df_flights.drop_duplicates()
df_flights = df_flights.dropna(subset=["price"])

df_flights.to_csv(os.path.join(CLEAN_DIR, "clean_flights.csv"), index=False, encoding="utf-8-sig")
print(f"Очищено:{df_flights.shape}")

In [ ]:
df_hotels = pd.read_csv(CSV_OUTPUT)
print(f"Сырых: {df_hotels.shape}")

df_hotels = df_hotels.replace(r'^\s*$', np.nan, regex=True)


def parse_distance_km(text):
    if pd.isna(text): return np.nan
    text = str(text)
    m = re.search(r'([\d.,]+)\s*km', text, re.IGNORECASE)
    if m: return float(m.group(1).replace(',', '.'))
    m = re.search(r'([\d.,]+)\s*m(?:\s|$)', text, re.IGNORECASE)
    if m: return round(float(m.group(1).replace(',', '.')) / 1000, 3)
    return np.nan


def parse_price(text):
    if pd.isna(text): return np.nan
    cleaned = re.sub(r'[€$£₽\s]', '', str(text)).replace(',', '')
    try: return float(cleaned)
    except ValueError:
        m = re.search(r'[\d]+\.?[\d]*', cleaned)
        return float(m.group(0)) if m else np.nan


def parse_rating(text):
    if pd.isna(text): return np.nan
    m = re.search(r'(\d+\.?\d*)', str(text))
    if m:
        val = float(m.group(1))
        return val if 0 <= val <= 10 else np.nan
    return np.nan


def parse_review_count(text):
    if pd.isna(text): return np.nan
    cleaned = re.sub(r'[^\d]', '', str(text))
    return int(cleaned) if cleaned else np.nan


df_hotels["distance_km"]= df_hotels["distance_to_center"].apply(parse_distance_km)
df_hotels["hotel_price"]= df_hotels["price"].apply(parse_price)
df_hotels["hotel_rating"] = df_hotels["rating_score"].apply(parse_rating)
df_hotels["hotel_reviews"] =df_hotels["review_count"].apply(parse_review_count)

df_hotels = df_hotels[[
    "city", "hotel_name", "url", "distance_km",
    "hotel_price", "hotel_rating", "hotel_reviews"
]].copy()

df_hotels= df_hotels.drop_duplicates(subset=["hotel_name", "city"])
df_hotels= df_hotels.dropna(subset=["hotel_price"])

df_hotels.to_csv(os.path.join(CLEAN_DIR, "clean_hotels.csv"), index=False, encoding="utf-8-sig")
print(f"Очищено:{df_hotels.shape}")

In [ ]:
df_grouped = pd.read_csv(os.path.join(RAW_DIR, "api_grouped_prices.csv"))
df_grouped["price"] = pd.to_numeric(df_grouped["price"], errors="coerce")
df_grouped = df_grouped.dropna(subset=["price"])

df_grouped.to_csv(os.path.join(CLEAN_DIR, "clean_grouped_prices.csv"),
                   index=False, encoding="utf-8-sig")
print(f"Grouped prices:{df_grouped.shape}")

---
# Часть 4. Подборки туров

In [ ]:
# маппинг код а потом название города (перелёты используют коды, отели — названия)
code_to_city= dict(zip(df_euro["destination_code"], df_euro["destination_name"]))

dest_col = "destination" if "destination" in df_flights.columns else "destination_code"
df_flights["city"] = df_flights[dest_col].map(code_to_city)

# каждый рейс на каждый отель в том же городе
df_tours = df_flights.merge(df_hotels, on="city", how="inner")

df_tours["flight_price"]= pd.to_numeric(df_tours["price"],errors="coerce")
df_tours["total_price"] = df_tours["flight_price"] + df_tours["hotel_price"]

print(f"Туров: {len(df_tours)},городов: {df_tours['city'].nunique()}, отелей: {df_tours['hotel_name'].nunique()}")

df_tours[["city", "hotel_name", "flight_price", "hotel_price", "total_price", "hotel_rating"]].head(10)

In [ ]:
# самая дешёвая дата вылета (из grouped_prices, API 5)

if not df_grouped.empty:
    cheapest = df_grouped.loc[df_grouped["price"].idxmin()]
    print(f"Самый дешёвый перелёт:")
    print(f"Направление: {cheapest.get('requested_destination', '?')}")
    print(f"Дата вылета:  {cheapest.get('group_key', '?')}")
    print(f"Цена:         {cheapest['price']} EUR")

    min_by_dest = df_grouped.groupby("requested_destination")["price"].min().sort_values()
    print(f"Минимальная цена по направлениям:")
    print(min_by_dest.to_string())
else:
    print("Нет данных grouped_prices")

In [ ]:
# TOп 10 самых дешёвых туров

top_cheap = df_tours.nsmallest(10, "total_price")
top_cheap[["city", "hotel_name", "flight_price", "hotel_price", "total_price", "hotel_rating"]]

In [ ]:
# TOП 10 лучшее соотношение цена/качество
# Добавить Полине нов функци о котор говорили 

df_vfm = df_tours[
    df_tours["hotel_rating"].notna() & (df_tours["total_price"] > 0)
].copy()
df_vfm["value_for_money"] = (df_vfm["hotel_rating"] / df_vfm["total_price"] * 1000).round(1)

top_vfm = df_vfm.nlargest(10, "value_for_money")
top_vfm[["city", "hotel_name", "hotel_rating", "total_price", "value_for_money"]]

In [ ]:
# средние показатели по городам

comparison = df_tours.groupby("city").agg(
    avg_total=("total_price", "mean"),
    avg_flight=("flight_price", "mean"),
    avg_hotel=("hotel_price", "mean"),
    avg_rating=("hotel_rating", "mean"),
    num_hotels=("hotel_name", "nunique"),
).round(1).sort_values("avg_total")

comparison.columns = ["Ср. тур (EUR)", "Ср. перелёт", "Ср. отель", "Ср. рейтинг", "Отелей"]
comparison